In [79]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import os

In [80]:
data_path = "../data/datasets/panel_dataset_VIF_normalized.xlsx"
df = pd.read_excel(data_path)
all_results = []
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [377]:
TARGETS = ['RECESS', 'RECESS_OVER', 'RECESS_PERIOD']
target_name = "RECESS_PERIOD"
exclude_cols = ['date', 'Country'] + TARGETS
feature_cols = [col for col in df.columns if col not in exclude_cols]

In [378]:
for country, group in df.groupby("Country"):
    group = group.reset_index(drop=True)

    # Decide whether to use month_sin/cos
    if country in ['JAP', 'GER']:
        exclude_cols = ['date', 'Country', 'RECESS', 'RECESS_PERIOD', 'RECESS_OVER', 'month', target_name]
    else:
        exclude_cols = ['date', 'Country', 'RECESS', 'RECESS_PERIOD', 'RECESS_OVER', 'month',
                        target_name, 'month_sin', 'month_cos']  # drop seasonal features

    feature_cols = [col for col in group.columns if col not in exclude_cols]

In [379]:
df.dropna(subset=feature_cols + [target_name], inplace=True)
df.sort_values(['Country', 'date'], inplace=True)

In [380]:
# Early stopping configuration
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

In [477]:
country = "JAP"
group = df[df["Country"] == country].copy().reset_index(drop=True)

if len(group) < 24:
    raise ValueError(f"Skipping {country}: insufficient data")


In [478]:
train_start = "1995-01-01"
train_end = "2009-12-31"
val_end = "2018-12-31"
test_end = "2024-05-31"

In [479]:
time_steps = 12
X_seq, y_seq, date_seq = [], [], []

for i in range(len(group) - time_steps):
    X_window = group.loc[i:i+time_steps-1, feature_cols].values
    y_value = group.loc[i + time_steps, target_name]
    date_value = group.loc[i + time_steps, 'date']

    X_seq.append(X_window)
    y_seq.append(y_value)
    date_seq.append(date_value)

X = np.array(X_seq)
y = np.array(y_seq)
dates = pd.to_datetime(date_seq)


In [480]:
train_mask = (dates >= train_start) & (dates <= train_end)
val_mask = (dates > train_end) & (dates <= val_end)
test_mask = (dates > val_end) & (dates <= test_end)


X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [481]:
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Train: 156 samples
Validation: 108 samples
Test: 65 samples


In [482]:
model_lstm = Sequential()
model_lstm.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
model_lstm.add(Dropout(0.3))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [483]:
history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - loss: 0.3959 - mae: 0.5042 - val_loss: 0.2359 - val_mae: 0.3289
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1368 - mae: 0.2885 - val_loss: 0.2519 - val_mae: 0.4083
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1217 - mae: 0.2893 - val_loss: 0.2951 - val_mae: 0.3962
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.1121 - mae: 0.2669 - val_loss: 0.2544 - val_mae: 0.4145
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0860 - mae: 0.2352 - val_loss: 0.2581 - val_mae: 0.4046
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0876 - mae: 0.2283 - val_loss: 0.2461 - val_mae: 0.3940
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0614 - mae: 0.1929 - val_loss: 0.2346 - val_mae: 0.3940
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0809 - mae: 0.2386 - val_loss: 0.2424 - val_mae: 0.3952
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - lo

In [484]:
y_pred_lstm = model_lstm.predict(X_test).flatten()

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step


In [485]:
threshold = 0.5
y_pred_lstm_binary = (y_pred_lstm >= threshold).astype(int)

auc_lstm = roc_auc_score(y_test, y_pred_lstm)
f1_lstm = f1_score(y_test, y_pred_lstm_binary)
precision_lstm = precision_score(y_test, y_pred_lstm_binary)
recall_lstm = recall_score(y_test, y_pred_lstm_binary)
accuracy_lstm = accuracy_score(y_test, y_pred_lstm_binary)

In [486]:
print(f"[RESULTS] {target_name} — Standard LSTM")
print(f"COUNTRY: {country}")
print(f"AUC:       {auc_lstm:.4f}")
print(f"F1:        {f1_lstm:.4f}")
print(f"Precision: {precision_lstm:.4f}")
print(f"Recall:    {recall_lstm:.4f}")
print(f"Accuracy:  {accuracy_lstm:.4f}")
print(f"Threshold: {threshold}")

[RESULTS] RECESS_PERIOD — Standard LSTM
COUNTRY: JAP
AUC:       0.8902
F1:        0.2857
Precision: 1.0000
Recall:    0.1667
Accuracy:  0.6923
Threshold: 0.5


In [487]:
all_results.append({
    "Country": country,
    "Architecture": "Standard LSTM",
    "Target": target_name,
    "AUC": auc_lstm,
    "F1": f1_lstm,
    "Precision": precision_lstm,
    "Recall": recall_lstm,
    "Accuracy": accuracy_lstm,
    "Threshold": threshold
})

In [488]:
model_bilstm = Sequential()
model_bilstm.add(Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train.shape[1], X_train.shape[2])))
model_bilstm.add(Dropout(0.4))
model_bilstm.add(Dense(1))
model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [489]:
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.3906 - mae: 0.5018 - val_loss: 0.2069 - val_mae: 0.3370
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1620 - mae: 0.3081 - val_loss: 0.2480 - val_mae: 0.4017
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.1169 - mae: 0.2563 - val_loss: 0.2578 - val_mae: 0.3942
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1040 - mae: 0.2577 - val_loss: 0.2333 - val_mae: 0.3920
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0823 - mae: 0.2398 - val_loss: 0.2223 - val_mae: 0.3883
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1084 - mae: 0.2505 - val_loss: 0.2289 - val_mae: 0.3911
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0750 - mae: 0.2203 - val_loss: 0.2441 - val_mae: 0.3869
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0864 - mae: 0.2134 - val_loss: 0.2162 - val_mae: 0.4022
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - lo

In [490]:
y_pred_bilstm = model_bilstm.predict(X_test).flatten()

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step


In [491]:
threshold = 0.5
y_pred_bilstm_binary = (y_pred_bilstm >= threshold).astype(int)

auc_bilstm = roc_auc_score(y_test, y_pred_bilstm)
f1_bilstm = f1_score(y_test, y_pred_bilstm_binary)
precision_bilstm = precision_score(y_test, y_pred_bilstm_binary)
recall_bilstm = recall_score(y_test, y_pred_bilstm_binary)
accuracy_bilstm = accuracy_score(y_test, y_pred_bilstm_binary)

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [492]:
print(f"[RESULTS] {target_name} — Bidirectional LSTM")
print(f"COUNTRY: {country}")
print(f"AUC:       {auc_bilstm:.4f}")
print(f"F1:        {f1_bilstm:.4f}")
print(f"Precision: {precision_bilstm:.4f}")
print(f"Recall:    {recall_bilstm:.4f}")
print(f"Accuracy:  {accuracy_bilstm:.4f}")
print(f"Threshold: {threshold}")


[RESULTS] RECESS_PERIOD — Bidirectional LSTM
COUNTRY: JAP
AUC:       0.8313
F1:        0.0000
Precision: 0.0000
Recall:    0.0000
Accuracy:  0.6308
Threshold: 0.5


In [493]:
all_results.append({
    "Country": country,
    "Architecture": "Bidirectional LSTM",
    "Target": target_name,
    "AUC": auc_bilstm,
    "F1": f1_bilstm,
    "Precision": precision_bilstm,
    "Recall": recall_bilstm,
    "Accuracy": accuracy_bilstm,
    "Threshold": threshold
})

In [494]:
model_stacked = Sequential()
model_stacked.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model_stacked.add(Dropout(0.3))
model_stacked.add(LSTM(32, return_sequences=False))
model_stacked.add(Dropout(0.3))
model_stacked.add(Dense(1))
model_stacked.compile(optimizer='adam', loss='mse', metrics=['mae'])

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [495]:
history_stacked = model_stacked.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.2882 - mae: 0.4562 - val_loss: 0.2044 - val_mae: 0.3538
Epoch 2/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.1687 - mae: 0.3272 - val_loss: 0.2470 - val_mae: 0.3521
Epoch 3/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1071 - mae: 0.2580 - val_loss: 0.2986 - val_mae: 0.3687
Epoch 4/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0942 - mae: 0.2429 - val_loss: 0.2677 - val_mae: 0.3678
Epoch 5/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0840 - mae: 0.2254 - val_loss: 0.2471 - val_mae: 0.3539
Epoch 6/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0711 - mae: 0.2010 - val_loss: 0.2249 - val_mae: 0.3623
Epoch 7/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0871 - mae: 0.2312 - val_loss: 0.2559 - val_mae: 0.3647
Epoch 8/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0743 - mae: 0.2054 - val_loss: 0.2283 - val_mae: 0.3700
Epoch 9/100
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - lo

In [496]:
y_pred_stacked_lstm = model_stacked.predict(X_test).flatten()

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step


In [497]:
threshold = 0.5
y_pred_stacked_lstm_binary = (y_pred_stacked_lstm >= threshold).astype(int)

auc_stacked_lstm = roc_auc_score(y_test, y_pred_stacked_lstm)
f1_stacked_lstm = f1_score(y_test, y_pred_stacked_lstm_binary)
precision_stacked_lstm = precision_score(y_test, y_pred_stacked_lstm_binary)
recall_stacked_lstm = recall_score(y_test, y_pred_stacked_lstm_binary)
accuracy_stacked_lstm = accuracy_score(y_test, y_pred_stacked_lstm_binary)

C:\Users\Owner\PycharmProjects\SURE_Program\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [498]:
print(f"[RESULTS] {target_name} — Stacked LSTM")
print(f"COUNTRY: {country}")
print(f"AUC:       {auc_stacked_lstm:.4f}")
print(f"F1:        {f1_stacked_lstm:.4f}")
print(f"Precision: {precision_stacked_lstm:.4f}")
print(f"Recall:    {recall_stacked_lstm:.4f}")
print(f"Accuracy:  {accuracy_stacked_lstm:.4f}")
print(f"Threshold: {threshold}")


[RESULTS] RECESS_PERIOD — Stacked LSTM
COUNTRY: JAP
AUC:       0.8516
F1:        0.0000
Precision: 0.0000
Recall:    0.0000
Accuracy:  0.6308
Threshold: 0.5


In [499]:
all_results.append({
    "Country": country,
    "Architecture": "Standard LSTM",
    "Target": target_name,
    "AUC": auc_stacked_lstm,
    "F1": f1_stacked_lstm,
    "Precision": precision_stacked_lstm,
    "Recall": recall_stacked_lstm,
    "Accuracy": accuracy_stacked_lstm,
    "Threshold": threshold
})


In [500]:
results_df = pd.DataFrame(all_results)
results_df

,Country,Architecture,Target,AUC,F1,Precision,Recall,Accuracy,Threshold
0,USA,Standard LSTM,RECESS,0.206349,0.000000,0.000000,0.000000,0.969231,0.5
1,USA,Bidirectional LSTM,RECESS,0.341270,0.000000,0.000000,0.000000,0.969231,0.5
2,USA,Standard LSTM,RECESS,0.365079,0.000000,0.000000,0.000000,0.723077,0.5
3,CAN,Standard LSTM,RECESS,0.239496,0.000000,0.000000,0.000000,0.784615,0.5
4,CAN,Bidirectional LSTM,RECESS,0.238095,0.000000,0.000000,0.000000,0.584615,0.5
5,CAN,Standard LSTM,RECESS,0.326331,0.000000,0.000000,0.000000,0.784615,0.5
6,MEX,Standard LSTM,RECESS,0.628676,0.000000,0.000000,0.000000,0.738462,0.5
7,MEX,Bidirectional LSTM,RECESS,0.890931,0.685714,0.666667,0.705882,0.830769,0.5
8,MEX,Standard LSTM,RECESS,0.987745,0.000000,0.000000,0.000000,0.738462,0.5
9,GER,Standard LSTM,RECESS,0.319853,0.000000,0.000000,0.000000,0.738462,0.5
